# 10 — Análisis jerárquico de agarres

**Pregunta:** ¿qué agarres se parecen fisiológicamente entre sí y cómo se organizan en familias?

1. Se calcula el **centroide** de cada agarre (perfil medio de las variables fusionadas).
2. Se mide la **distancia de Mahalanobis** entre centroides, con la covarianza **dentro de los agarres**
   estimada con **Ledoit-Wolf**.
3. Con esas distancias se construye un **dendrograma** de agarres.
4. Se comprueba la **estabilidad** del dendrograma remuestreando sujetos (bootstrap).

Para usar otro dataset solo se cambian `DATA_PATH` y `FEATURE_COLS`.

| Sección | Contenido |
|---|---|
| 1 | Carga (sin reposo, sin reescalar) |
| 2 | Centroides por agarre |
| 3 | Covarianza Ledoit-Wolf y distancia de Mahalanobis |
| 4 | Dendrograma y elección del método de enlace |
| 5 | Estabilidad: bootstrap por sujetos |
| 6 | Resumen y guardado |

In [ ]:
# =====================================================================
# PARÁMETROS DEL DATASET — lo único que hay que cambiar para otro dataset
# =====================================================================
DATA_PATH = '../data/master/intact_bc_fused.csv'
FEATURE_COLS = ['VAR_STD', 'Range', 'Energy', 'Peak_to_Peak', 'VAR', 'Max', 'MNF', 'SSC']
# =====================================================================

In [ ]:
# ---------------- Configuración (normalmente no se toca) ----------------
LABEL_COL, SUBJECT_COL, DB_COL = 'stimulus', 'subject', 'origin_db'
REST = 0
LINKAGE_METHODS = ['average', 'complete', 'ward']   # se comparan en la sección 4
N_BOOT = 500                                         # remuestreos de sujetos (sección 5)
RANDOM_STATE = 42
# -------------------------------------------------------------------------

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.covariance import LedoitWolf
from scipy.cluster.hierarchy import linkage, dendrogram, cophenet, to_tree
from scipy.spatial.distance import squareform

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

N_FEATURES = len(FEATURE_COLS)
DATASET_TAG = f'{Path(DATA_PATH).stem}__{N_FEATURES}f'
SAVE_DIR = f'../figures/jerarquico/{DATASET_TAG}'
RESULTS_DIR = f'../results/jerarquico/{DATASET_TAG}'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Dataset: {DATA_PATH} ({N_FEATURES} variables)')
print(f'Figuras: {SAVE_DIR}\nResultados: {RESULTS_DIR}')


def savefig(fig, name):
    fig.savefig(f'{SAVE_DIR}/{name}.pdf', dpi=300, bbox_inches='tight')

## 1. Carga

El dataset ya viene transformado y normalizado: **no se vuelve a escalar**. Se quita el reposo, porque es
trivialmente distinto de todos los agarres y dominaría el dendrograma (quedaría solo en una rama aparte y
comprimiría las distancias entre los agarres reales).

In [ ]:
df = pd.read_csv(DATA_PATH)
missing = [c for c in [LABEL_COL, SUBJECT_COL, DB_COL] + FEATURE_COLS if c not in df.columns]
if missing:
    raise KeyError(f'Faltan columnas en el CSV: {missing}')

n_rest = (df[LABEL_COL] == REST).sum()
df = df[df[LABEL_COL] != REST].copy().reset_index(drop=True)
df['group'] = df[SUBJECT_COL].astype(str) + '_' + df[DB_COL].astype(str)

X = df[FEATURE_COLS].values.astype(float)
y = df[LABEL_COL].values
groups = df['group'].values

STIMULI = sorted(np.unique(y).tolist())
LABELS = [f'Agarre {s}' for s in STIMULI]
N_STIM = len(STIMULI)
print(f'Filas de reposo eliminadas: {n_rest}')
print(f'Filas: {len(df)} | agarres ({N_STIM}): {STIMULI} | sujetos: {len(np.unique(groups))}')

## 2. Centroides por agarre

El centroide de un agarre es el valor medio de cada variable sobre todos los sujetos. El heatmap muestra el
**perfil** de cada agarre: qué variables están por encima o por debajo de la media. Sirve para interpretar
fisiológicamente el dendrograma (por qué dos agarres quedan juntos).

In [ ]:
def class_centroids(X, y):
    return np.vstack([X[y == s].mean(axis=0) for s in STIMULI])


centroids = class_centroids(X, y)
centroids_df = pd.DataFrame(centroids, index=LABELS, columns=FEATURE_COLS)

fig, ax = plt.subplots(figsize=(1.0 * N_FEATURES + 3, 0.55 * N_STIM + 1.8), dpi=300)
fig.patch.set_facecolor('#F7F9FC')
lim = np.abs(centroids).max()
sns.heatmap(centroids_df, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-lim, vmax=lim,
            linewidths=0.5, linecolor='#FFFFFF', cbar_kws={'label': 'Valor medio (normalizado)'}, ax=ax)
ax.set_title('Centroides por agarre (perfil medio de cada variable)', fontweight='bold')
ax.set_xlabel('Variable', fontweight='bold')
ax.set_ylabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
savefig(fig, 'centroides')
plt.show()

## 3. Covarianza Ledoit-Wolf y distancia de Mahalanobis

**Por qué Mahalanobis y no euclídea:** 6 de las 8 variables son de amplitud y están muy correlacionadas.
En la distancia euclídea la amplitud contaría casi 6 veces. Mahalanobis corrige eso usando la covarianza:
las variables redundantes cuentan una sola vez y una diferencia pesa más si ocurre en una dirección
donde los sujetos suelen variar poco.

**Qué covarianza se usa:** la covarianza **dentro de los agarres** (*pooled*): se resta a cada fila el
centroide de su agarre y se calcula la covarianza de esos residuos. Así se mide cuánto varían los sujetos
dentro de un mismo agarre. Si se usara la covarianza del dataset completo, incluiría las propias diferencias
entre agarres y encogería las distancias que se quieren medir.

**Por qué Ledoit-Wolf:** con variables casi colineales la covarianza normal es casi singular y su inversa
es inestable. Ledoit-Wolf la "encoge" hacia una matriz diagonal en la cantidad óptima (`shrinkage`), lo que
hace la inversa estable sin elegir un valor a mano.

In [ ]:
def pooled_lw(X, y):
    # Covarianza dentro de los agarres con Ledoit-Wolf
    resid = X - np.vstack([X[y == s].mean(axis=0) for s in STIMULI])[np.searchsorted(STIMULI, y)]
    return LedoitWolf().fit(resid)


def mahalanobis_matrix(centroids, precision):
    diff = centroids[:, None, :] - centroids[None, :, :]
    d2 = np.einsum('ijk,kl,ijl->ij', diff, precision, diff)
    D = np.sqrt(np.maximum(d2, 0))
    np.fill_diagonal(D, 0)
    return D


lw = pooled_lw(X, y)
precision = lw.precision_
cov_emp = np.cov(X - centroids[np.searchsorted(STIMULI, y)], rowvar=False)

print(f'Shrinkage Ledoit-Wolf: {lw.shrinkage_:.3f}  (0 = covarianza empírica, 1 = totalmente diagonal)')
print(f'Número de condición — empírica: {np.linalg.cond(cov_emp):,.0f} | '
      f'Ledoit-Wolf: {np.linalg.cond(lw.covariance_):,.0f}')

D = mahalanobis_matrix(centroids, precision)
D_df = pd.DataFrame(D, index=LABELS, columns=LABELS)

pairs = [{'agarre_a': STIMULI[i], 'agarre_b': STIMULI[j], 'mahalanobis': D[i, j]}
         for i in range(N_STIM) for j in range(i + 1, N_STIM)]
pairs_df = pd.DataFrame(pairs).sort_values('mahalanobis').reset_index(drop=True)
pairs_df.index = pairs_df.index + 1
print('\nPares más parecidos (menor distancia):')
display(pairs_df.head(5).round(3))
print('Pares más distintos (mayor distancia):')
display(pairs_df.tail(3).round(3))

## 4. Dendrograma

El dendrograma une primero los agarres más parecidos y luego va juntando grupos. La altura de cada unión es
la distancia a la que se juntan: uniones bajas = agarres muy parecidos.

**Método de enlace** (cómo se mide la distancia entre dos grupos ya formados):
- `average`: media de las distancias entre sus miembros (UPGMA). Es el más usado con matrices de distancia.
- `complete`: la distancia máxima entre sus miembros.
- `ward`: minimiza la varianza dentro de los grupos. Es válido aquí porque la distancia de Mahalanobis es una
  distancia euclídea en el espacio "blanqueado" por la covarianza.

Se elige el método con mayor **correlación cofenética**: cuánto respeta el dendrograma las distancias
originales (1 = perfectamente).

In [ ]:
condensed = squareform(D, checks=False)
coph = {}
for m in LINKAGE_METHODS:
    Zm = linkage(condensed, method=m)
    coph[m] = cophenet(Zm, condensed)[0]
coph_df = pd.Series(coph, name='correlación cofenética').to_frame().round(3)
display(coph_df)

LINKAGE = max(coph, key=coph.get)
Z = linkage(condensed, method=LINKAGE)
print(f'Método elegido: {LINKAGE} (cofenética = {coph[LINKAGE]:.3f})')

In [ ]:
# Heatmap de distancias ordenado según el dendrograma
order = dendrogram(Z, no_plot=True, labels=LABELS)['ivl']
D_ord = D_df.loc[order, order]

fig, ax = plt.subplots(figsize=(0.9 * N_STIM + 3, 0.8 * N_STIM + 1.5), dpi=300)
fig.patch.set_facecolor('#F7F9FC')
mask = np.eye(N_STIM, dtype=bool)
sns.heatmap(D_ord, annot=True, fmt='.2f', cmap='mako_r', mask=mask,
            linewidths=0.5, linecolor='#FFFFFF', cbar_kws={'label': 'Distancia de Mahalanobis'}, ax=ax)
ax.set_title('Distancia de Mahalanobis entre centroides\n(ordenada según el dendrograma)', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
savefig(fig, 'mahalanobis_heatmap')
plt.show()

## 5. Estabilidad del dendrograma (bootstrap por sujetos)

Un dendrograma siempre se puede dibujar, aunque las diferencias sean ruido. Para saber qué uniones son
fiables se repite todo el proceso `N_BOOT` veces con una muestra de sujetos tomada **con reemplazo**
(centroides, covarianza Ledoit-Wolf, distancias y dendrograma).

- **Soporte de cada rama:** % de remuestreos en que aparece exactamente ese grupo de agarres.
  Se anota en el dendrograma. Como referencia: ≥ 70 % es una agrupación sólida; < 50 %, poco fiable.
- **Probabilidad de co-agrupación:** para cada par de agarres, % de remuestreos en que quedan en el mismo
  grupo al cortar el dendrograma en k grupos.

In [ ]:
def clades(Z, n):
    # Conjuntos de hojas de cada rama interna (sin la raíz)
    root = to_tree(Z)
    out = []
    def walk(node):
        if node.is_leaf():
            return frozenset([node.id])
        s = walk(node.left) | walk(node.right)
        if len(s) < n:
            out.append(s)
        return s
    walk(root)
    return set(out)


rng = np.random.default_rng(RANDOM_STATE)
uniq_groups = np.unique(groups)
idx_by_group = {g: np.where(groups == g)[0] for g in uniq_groups}

ref_clades = clades(Z, N_STIM)
clade_hits = {c: 0 for c in ref_clades}
D_boot = []
Z_boot = []
n_ok = 0
for b in range(N_BOOT):
    sample = rng.choice(uniq_groups, size=len(uniq_groups), replace=True)
    idx = np.concatenate([idx_by_group[g] for g in sample])
    Xb, yb = X[idx], y[idx]
    if len(np.unique(yb)) < N_STIM:
        continue
    cb = class_centroids(Xb, yb)
    Db = mahalanobis_matrix(cb, pooled_lw(Xb, yb).precision_)
    Zb = linkage(squareform(Db, checks=False), method=LINKAGE)
    for c in clades(Zb, N_STIM) & ref_clades:
        clade_hits[c] += 1
    D_boot.append(Db)
    Z_boot.append(Zb)
    n_ok += 1

support = {c: 100 * h / n_ok for c, h in clade_hits.items()}
D_boot = np.array(D_boot)
D_lo, D_hi = np.percentile(D_boot, [2.5, 97.5], axis=0)
print(f'Remuestreos válidos: {n_ok} de {N_BOOT}')

support_df = (pd.DataFrame([{'grupo': ' + '.join(str(STIMULI[i]) for i in sorted(c)),
                             'n_agarres': len(c), 'soporte_%': s} for c, s in support.items()])
              .sort_values(['n_agarres', 'soporte_%'], ascending=[True, False]).reset_index(drop=True))
display(support_df.round(1))

In [ ]:
# Dendrograma final con el soporte bootstrap en cada unión
fig, ax = plt.subplots(figsize=(11, 6), dpi=300)
fig.patch.set_facecolor('#F7F9FC')
ax.set_facecolor('#FFFFFF')
dn = dendrogram(Z, labels=LABELS, leaf_font_size=11, color_threshold=0, above_threshold_color='#37474F',
                ax=ax)

# Posición (x, altura) de cada unión → su conjunto de hojas
node_sets = {}
for i, (a, b, h, _) in enumerate(Z):
    def members(k):
        return {int(k)} if k < N_STIM else node_sets[int(k) - N_STIM]
    node_sets[i] = members(a) | members(b)

# Posición horizontal de cada nodo: hojas en 5, 15, 25…; un nodo queda en el punto medio de sus hijos
node_x = {LABELS.index(lbl): 5 + 10 * k for k, lbl in enumerate(dn['ivl'])}
for i, (a, b, h, _) in enumerate(Z):
    node_x[N_STIM + i] = (node_x[int(a)] + node_x[int(b)]) / 2

for i, (a, b, h, _) in enumerate(Z):
    s = frozenset(node_sets[i])
    if len(s) == N_STIM:
        continue
    x = node_x[N_STIM + i]
    sup = support.get(s, 0)
    color = '#2E7D32' if sup >= 70 else ('#F9A825' if sup >= 50 else '#C62828')
    ax.text(x, h, f'{sup:.0f}%', ha='center', va='bottom', fontsize=8, fontweight='bold', color=color,
            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec=color, lw=0.8))

ax.set_ylabel('Distancia de Mahalanobis', fontweight='bold')
ax.set_title(f'Dendrograma de agarres — Mahalanobis (Ledoit-Wolf) + enlace {LINKAGE}\n'
             f'% = soporte bootstrap por sujetos ({n_ok} remuestreos; verde ≥ 70 %, rojo < 50 %)',
             fontweight='bold', fontsize=11)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
savefig(fig, 'dendrograma')
plt.show()

In [ ]:
# Probabilidad de que cada par de agarres quede en el mismo grupo, para varios cortes del dendrograma
from scipy.cluster.hierarchy import fcluster

K_CUTS = [k for k in (2, 3, 4) if k < N_STIM]
fig, axes = plt.subplots(1, len(K_CUTS), figsize=(5.2 * len(K_CUTS), 4.6), dpi=300)
axes = np.atleast_1d(axes)
fig.patch.set_facecolor('#F7F9FC')
coassign = {}
for ax, k in zip(axes, K_CUTS):
    P = np.zeros((N_STIM, N_STIM))
    for Zb in Z_boot:
        lab = fcluster(Zb, t=k, criterion='maxclust')
        P += (lab[:, None] == lab[None, :])
    P = 100 * P / len(Z_boot)
    coassign[k] = P
    P_df = pd.DataFrame(P, index=LABELS, columns=LABELS).loc[order, order]
    sns.heatmap(P_df, annot=True, fmt='.0f', cmap='Greens', vmin=0, vmax=100, cbar=False,
                linewidths=0.5, linecolor='#FFFFFF', ax=ax, annot_kws={'fontsize': 7})
    ax.set_title(f'Corte en k = {k} grupos', fontweight='bold')
    ax.tick_params(labelsize=7)
fig.suptitle('% de remuestreos en que dos agarres quedan en el mismo grupo', fontweight='bold')
plt.tight_layout()
savefig(fig, 'coagrupacion_bootstrap')
plt.show()

## 6. Resumen y guardado

In [ ]:
pairs_df['IC95_inf'] = [D_lo[STIMULI.index(a), STIMULI.index(b)] for a, b in zip(pairs_df.agarre_a, pairs_df.agarre_b)]
pairs_df['IC95_sup'] = [D_hi[STIMULI.index(a), STIMULI.index(b)] for a, b in zip(pairs_df.agarre_a, pairs_df.agarre_b)]

centroids_df.to_csv(f'{RESULTS_DIR}/centroides.csv')
D_df.to_csv(f'{RESULTS_DIR}/mahalanobis.csv')
pairs_df.to_csv(f'{RESULTS_DIR}/pares_mahalanobis.csv', index=False)
support_df.to_csv(f'{RESULTS_DIR}/soporte_ramas.csv', index=False)

print(f'Configuración: {N_FEATURES} variables, enlace {LINKAGE} (cofenética {coph[LINKAGE]:.3f}), '
      f'shrinkage LW {lw.shrinkage_:.3f}, {n_ok} remuestreos\n')
print('Pares más parecidos (Mahalanobis, IC 95 % bootstrap):')
for r in pairs_df.head(3).itertuples():
    print(f'  {r.agarre_a}–{r.agarre_b}: {r.mahalanobis:.2f} [{r.IC95_inf:.2f}, {r.IC95_sup:.2f}]')
print('Pares más distintos:')
for r in pairs_df.tail(3).itertuples():
    print(f'  {r.agarre_a}–{r.agarre_b}: {r.mahalanobis:.2f} [{r.IC95_inf:.2f}, {r.IC95_sup:.2f}]')
print('\nRamas con soporte ≥ 70 %:')
solid = support_df[support_df['soporte_%'] >= 70]
print('  ' + ('\n  '.join(f'{{{r.grupo}}} ({r["soporte_%"]:.0f} %)' for _, r in solid.iterrows())
              if len(solid) else 'ninguna'))
print(f'\nResultados guardados en {RESULTS_DIR}')

## 7. Interpretación

*(Completar con los resultados.)*

- **Familias de agarres:** ramas con soporte alto (≥ 70 %). Usar el heatmap de centroides para explicar qué
  variables comparten (p. ej., agarres de mucha fuerza vs. agarres de precisión).
- **Contraste con la separabilidad:** los pares más cercanos aquí deberían coincidir con los pares más
  confundidos de la LDA en LOSO. Si coinciden, las dos vistas (supervisada y de distancias) cuentan la misma
  historia.
- **Ramas con soporte bajo:** el orden de esas uniones depende de qué sujetos entren en la muestra; no
  conviene interpretarlas.

**Limitaciones**
- Cada sujeto aporta una sola fila por agarre (media de sus repeticiones), así que el análisis describe el
  perfil medio de los agarres, no la variabilidad dentro de cada sujeto.
- La distancia de Mahalanobis supone una covarianza común a todos los agarres (pooled).